# 🗄️ Vector Stores

This notebook walks through practical vector store usage with **Chroma**, covering
document ingestion, similarity search (with and without scores), metadata filtering,
retriever configuration, and persistence to disk. It closes with an exercise that
builds a reusable chunk-and-retrieve pipeline from raw text.

## Learning Objectives
In this notebook, you will learn:
1. **Vector store basics** - creating a Chroma collection from `Document` objects and running similarity search
2. **Scored retrieval** - interpreting distance scores returned alongside search results
3. **Metadata filtering** - narrowing similarity search to documents matching metadata criteria
4. **Retrievers** - wrapping a vector store as a `Retriever`, including MMR-based diverse retrieval
5. **Persistence** - saving a Chroma collection to disk and reloading it in a new session

## Prerequisites
- An `OPENAI_API_KEY` set in a `.env` file at the project root (used for embeddings)
- `langchain-openai`, `langchain-chroma`, `langchain-core`, and `langchain-text-splitters` installed
- Familiarity with `Document` objects and basic LangChain concepts


---
## 🔧 Part 1: Setup

We load environment variables from `.env`, initialize the embedding model used
throughout the notebook, and define a small corpus of sample documents (with
`source` and `topic` metadata) that every demo below indexes into Chroma.


In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Imports and Embedding Model
# ============================================================================
import shutil
import tempfile

from dotenv import load_dotenv

from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()

embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")

print("✅ Environment loaded and embedding model initialized (text-embedding-3-small).")


### 📄 Sample Corpus

A small set of `Document` objects with `source` and `topic` metadata, reused as
the shared corpus across every demo in this notebook.


In [ ]:
# ============================================================================
# SAMPLE DOCUMENTS: Shared Corpus for Vector Store Demos
# ============================================================================
SAMPLE_DOCS = [
    Document(
        page_content="LangChain is a framework for developing applications powered by language models.",
        metadata={"source": "langchain_docs", "topic": "overview"},
    ),
    Document(
        page_content="LangGraph is a library for building stateful, multi-actor applications with LLMs.",
        metadata={"source": "langgraph_docs", "topic": "overview"},
    ),
    Document(
        page_content="Vector stores are databases optimized for storing and searching embeddings.",
        metadata={"source": "vector_guide", "topic": "database"},
    ),
    Document(
        page_content="RAG combines retrieval with generation for more accurate LLM responses.",
        metadata={"source": "rag_guide", "topic": "architecture"},
    ),
    Document(
        page_content="Embeddings convert text into numerical vectors for semantic similarity.",
        metadata={"source": "embeddings_guide", "topic": "fundamentals"},
    ),
    Document(
        page_content="Chroma is an open-source embedding database for AI applications.",
        metadata={"source": "chroma_docs", "topic": "database"},
    ),
    Document(
        page_content="FAISS is a library for efficient similarity search developed by Facebook.",
        metadata={"source": "faiss_docs", "topic": "database"},
    ),
    Document(
        page_content="Pinecone is a managed vector database service for production workloads.",
        metadata={"source": "pinecone_docs", "topic": "database"},
    ),
]

print(f"✅ Sample corpus ready: {len(SAMPLE_DOCS)} documents.")


---
## 🔍 Part 2: Chroma Basics

The core Chroma workflow: build a vector store from `Document` objects with
`Chroma.from_documents`, persist it to a temporary directory, and run a plain
similarity search against it.


In [ ]:
# ============================================================================
# CHROMA_BASICS: Build a Vector Store and Run Similarity Search
# ============================================================================
def chroma_basics():
    with tempfile.TemporaryDirectory() as tmpdir:
        # create vector store from documents
        vectorstore = Chroma.from_documents(
            documents=SAMPLE_DOCS, embedding=embeddings_model, persist_directory=tmpdir
        )
        print(
            f"Vector store created {vectorstore._collection.count()} documents and persisted."
        )

        # perform similarity search
        query = "What is LangChain?"
        results = vectorstore.similarity_search(query, k=2)

        print(f"Top 2 results for query '{query}':")
        for i, doc in enumerate(results):
            print(
                f"Result {i+1}: {doc.page_content} (Source: {doc.metadata['source']})"
            )


---
## 🧮 Part 3: Similarity Search with Scores

`similarity_search_with_score` returns each document alongside a raw distance
score. Chroma's default distance metric means *lower* is more similar, so we
convert it to an intuitive 0-1 similarity value with `1 / (1 + distance)`.


In [ ]:
# ============================================================================
# SIMILARITY_SEARCH_WITH_SCORES: Retrieve Documents with Distance Scores
# ============================================================================
def similarity_search_with_scores():
    with tempfile.TemporaryDirectory() as tmpdir:
        # create vector store from documents
        vectorstore = Chroma.from_documents(
            documents=SAMPLE_DOCS, embedding=embeddings_model, persist_directory=tmpdir
        )

        # perform similarity search with scores
        query = "Explain vector stores."
        results_with_scores = vectorstore.similarity_search_with_score(query, k=3)

        print(f"Top 3 results with scores for query '{query}':")
        for i, (doc, score) in enumerate(results_with_scores):
            final_score = 1 / (1 + score)  # Convert distance to similarity
            print(
                f"Result {i+1}: {doc.page_content} (Score: {final_score:.4f}, Source: {doc.metadata['source']})"
            )


---
## 🏷️ Part 4: Metadata Filtering

Similarity search can be narrowed to documents whose metadata matches a filter
(e.g. `{"topic": "database"}`), letting you combine semantic relevance with
exact structured constraints. This cell contrasts results with and without the
filter applied to the same query.


In [ ]:
# ============================================================================
# METADATA_FILTERING: Similarity Search With and Without Metadata Filters
# ============================================================================
def metadata_filtering():
    with tempfile.TemporaryDirectory() as tmpdir:
        # create vector store from documents
        vectorstore = Chroma.from_documents(
            documents=SAMPLE_DOCS, embedding=embeddings_model, persist_directory=tmpdir
        )

        query = "What databases are available?"

        # without metadata filtering
        results = vectorstore.similarity_search(query, k=5)
        print(f"Results without metadata filtering for query '{query}':")
        for i, doc in enumerate(results):
            print(
                f"Result {i+1}: {doc.page_content} (Source: {doc.metadata['source']})"
            )

        # with metadata filtering
        filter_criteria = {"topic": "database"}
        filtered_results = vectorstore.similarity_search(
            query, k=5, filter=filter_criteria
        )
        print(f"\nResults with metadata filtering for query '{query}':")
        for i, doc in enumerate(filtered_results):
            print(
                f"Result {i+1}: {doc.page_content} (Source: {doc.metadata['source']})"
            )


---
## 🔗 Part 5: Retrievers

A vector store can be wrapped as a `Retriever` via `as_retriever`, which
standardizes it behind the `invoke` interface used across LangChain pipelines.
This section compares plain `similarity` search against `mmr` (Maximal
Marginal Relevance), which trades off relevance for diversity among results.

### Key Concepts:
- **`search_type="similarity"`**: returns the `k` most similar documents.
- **`search_type="mmr"`**: fetches `fetch_k` candidates, then selects `k` of
  them balancing relevance and diversity.


In [ ]:
# ============================================================================
# AS_RETRIEVER: Wrap a Vector Store as a Retriever (Similarity vs. MMR)
# ============================================================================
def as_retriever():

    with tempfile.TemporaryDirectory() as tmpdir:
        vectorstore = Chroma.from_documents(
            documents=SAMPLE_DOCS,
            embedding=OpenAIEmbeddings(model="text-embedding-3-small"),
            persist_directory=tmpdir,
        )

        # basic retriever usage
        retriever = vectorstore.as_retriever(
            search_type="similarity", search_kwargs={"k": 3}
        )
        # use retriever to get relevant documents
        docs = retriever.invoke("How do I build AI applications?")

        print("Retriever results:")
        for i, doc in enumerate(docs):
            print(
                f"Result {i+1}: {doc.page_content} (Source: {doc.metadata['source']})"
            )

        mmr_retriever = vectorstore.as_retriever(
            search_type="mmr",
            search_kwargs={"k": 3, "fetch_k": 5},  # fetch 5 docs and return 3 diverse
        )
        mmr_docs = mmr_retriever.invoke("vector databases and embeddings")
        print("\nMMR Retriever results:")
        for i, doc in enumerate(mmr_docs):
            print(
                f"Result {i+1}: {doc.page_content} (Source: {doc.metadata['source']})"
            )


---
## 💾 Part 6: Persisting and Reloading Chroma

Unlike the previous demos (which use a temporary directory that is deleted on
exit), this section persists a Chroma collection to a real on-disk directory
(`./chroma_db/`), deletes the in-memory reference to simulate a process
restart, then reloads the collection from disk and confirms search still
works.

> **Note**: Running this cell creates a `./chroma_db/` directory in the
> notebook's working directory. Delete it manually if you want a clean slate.


In [ ]:
# ============================================================================
# PERSIST_CHROMA: Persist to Disk, Simulate a Restart, and Reload
# ============================================================================
def persist_chroma():
    persist_dir = "./chroma_db/"

    vectorstore = Chroma.from_documents(
        documents=SAMPLE_DOCS,
        embedding=embeddings_model,
        persist_directory=persist_dir,
    )

    original_count = vectorstore._collection.count()
    print(f"Persisted vector store with {original_count} documents.")
    print(f"Vector store persisted at: {persist_dir}")

    # simulate restart - load from disk
    del vectorstore

    reloaded = Chroma(
        embedding_function=embeddings_model,
        persist_directory=persist_dir,
    )

    reloaded_count = reloaded._collection.count()
    print(f"Reloaded vector store with {reloaded_count} documents.")

    # verify search still works
    results = reloaded.similarity_search("LangChain", k=2)
    print(f"Search result: {results[0].page_content[:50]}...")


---
## 📝 Part 7: Exercise — End-to-End Chunk-and-Retrieve Pipeline

**Exercise**: build a complete vector store pipeline that takes a list of raw
text strings, splits them into chunks with `RecursiveCharacterTextSplitter`,
stores the chunks in an in-memory Chroma collection, and returns a configured
retriever. The cell below defines `create_retriever` and tests it against two
queries over a small multi-language programming corpus.


In [ ]:
# ============================================================================
# EXERCISE_VECTOR_STORE_SETUP: Chunk, Store, and Return a Retriever
# ============================================================================
def exercise_vector_store_setup():
    """
    EXERCISE: Create a complete vector store setup that:
    1. Takes a list of text strings
    2. Splits them into chunks
    3. Stores in Chroma
    4. Returns a configured retriever

    Test with sample documents.
    """

    def create_retriever(
        texts: list[str], chunk_size: int = 500, chunk_overlap: int = 50, k: int = 3
    ):

        # Create documents
        docs = [Document(page_content=t) for t in texts]

        # Split
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size, chunk_overlap=chunk_overlap
        )
        split_docs = splitter.split_documents(docs)

        # Create vector store (in-memory for exercise)
        vectorstore = Chroma.from_documents(
            documents=split_docs, embedding=embeddings_model
        )

        # Return retriever
        return vectorstore.as_retriever(
            search_type="similarity", search_kwargs={"k": k}
        )

    # Test the function
    sample_texts = [
        "Python is a versatile programming language used in web development, "
        "data science, machine learning, and automation. It has a simple syntax "
        "that makes it easy to learn and read.",
        "JavaScript is the language of the web. It runs in browsers and on "
        "servers with Node.js. Modern frameworks like React and Vue make "
        "building web applications efficient.",
        "Rust is a systems programming language focused on safety and "
        "performance. It prevents common bugs like null pointer dereferences "
        "and data races at compile time.",
    ]

    retriever = create_retriever(sample_texts, chunk_size=200, chunk_overlap=20, k=2)

    print("Testing retriever:\n")
    queries = [
        "What's good for web development?",
        "Which language is safest?",
    ]
    for query in queries:
        print(f"Query: {query}")
        results = retriever.invoke(query)
        for doc in results:
            print(f"  - {doc.page_content[:60]}...")
        print()


---
## ▶️ Part 8: Run

The original `__main__` guard, kept verbatim. Jupyter sets `__name__` to
`"__main__"`, so this cell runs as-is. Uncomment a line to run that demo (only
one demo is uncommented by default to avoid rebuilding multiple vector stores).


In [ ]:
# ============================================================================
# RUN: Demo Entry Point
# ============================================================================
if __name__ == "__main__":
    # chroma_basics()
    # similarity_search_with_scores()
    # metadata_filtering()
    # as_retriever()
    # persist_chroma()
    exercise_vector_store_setup()


---
## 📝 Summary

### 1. Core Vector Store Operations
- **`Chroma.from_documents`**: builds a persisted or in-memory Chroma collection directly from `Document` objects and an embedding model.
- **`similarity_search`**: returns the `k` most semantically similar documents to a query.
- **`similarity_search_with_score`**: also returns a distance score per result; convert it to a similarity value (e.g. `1 / (1 + distance)`) for readability.
- **Metadata filtering**: pass `filter={"field": value}` to `similarity_search` to constrain results to documents matching exact metadata.

### 2. Retrievers and Persistence
- **`as_retriever`**: wraps a vector store behind the standard `Retriever.invoke` interface, configurable via `search_type` (`"similarity"` or `"mmr"`) and `search_kwargs`.
- **MMR (Maximal Marginal Relevance)**: fetches `fetch_k` candidates and selects `k` of them to balance relevance with diversity, useful when top results are near-duplicates.
- **Persistence**: passing `persist_directory` to `Chroma.from_documents` writes the collection to disk; re-instantiating `Chroma(embedding_function=..., persist_directory=...)` reloads it in a new process.

### Functions Defined in This Notebook
- `chroma_basics()`
- `similarity_search_with_scores()`
- `metadata_filtering()`
- `as_retriever()`
- `persist_chroma()`
- `exercise_vector_store_setup()`

### Next Steps
- Explore other vector store backends (FAISS, Pinecone) referenced in the sample corpus's own documents.
- Combine metadata filtering with MMR retrieval in a single retriever configuration.
- Move on to query transformation techniques (multi-query, HyDE, reranking) that sit on top of a retriever like the ones built here.
